In [19]:
# This practise is about using Random Forest to fill in missing values

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [45]:
data = fetch_california_housing()
data

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [18]:
# It's going to be a regresion model
data.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [17]:

dataset = pd.DataFrame(data.data,columns = data.feature_names)
dataset

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32


In [56]:
# No null values for this dataset...therefore I need to create some
dataset.isna().sum()

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64

In [27]:
data.data.shape

(20640, 8)

In [24]:
xfull,yfull = data.data,data.target
n_samples = xfull.shape[0]
n_features = xfull.shape[1]

In [30]:
#Firstly , determine the ratio of missing values, in this case I would do 50%
rng = np.random.RandomState(0)
missrate = 0.5
n_missing_samples = int(np.floor(n_samples*n_features*missrate))
n_missing_samples

82560

In [32]:
missing_features = rng.randint(0,n_features,n_missing_samples)
missing_samples = rng.randint(0,n_samples,n_missing_samples)

In [43]:
# Missing values are created across the dataset
x_missing = xfull.copy()
y_missing = yfull.copy()

x_missing[missing_samples,missing_features] = np.nan
pd.DataFrame(x_missing)

,0,1,2,3,4,5,6,7
0,8.3252,NaN,6.984127,NaN,NaN,2.555556,37.88,NaN
1,8.3014,NaN,NaN,0.971880,2401.0,2.109842,NaN,-122.22
2,NaN,NaN,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,NaN,5.817352,1.073059,NaN,2.547945,37.85,-122.25
4,3.8462,NaN,NaN,1.081081,NaN,NaN,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,NaN,2.560606,NaN,NaN
20636,NaN,18.0,NaN,1.315789,356.0,3.122807,NaN,-121.21
20637,NaN,17.0,5.205543,1.120092,1007.0,2.325635,39.43,NaN
20638,1.8672,NaN,NaN,1.171920,741.0,NaN,39.43,-121.32


In [61]:
# Use SimpleImputer to fill 
imp_mean = SimpleImputer(missing_values=np.nan,strategy="mean")
# Prepare X mean value
x_missing_mean = imp_mean.fit_transform(x_missing)

In [64]:
# Filling numers
pd.DataFrame(x_missing_mean)

,0,1,2,3,4,5,6,7
0,8.325200,28.616222,6.984127,1.100913,1419.802875,2.555556,37.880000,-119.57379
1,8.301400,28.616222,5.433694,0.971880,2401.000000,2.109842,35.642578,-122.22000
2,3.850569,28.616222,8.288136,1.073446,496.000000,2.802260,37.850000,-122.24000
3,5.643100,28.616222,5.817352,1.073059,1419.802875,2.547945,37.850000,-122.25000
4,3.846200,28.616222,5.433694,1.081081,1419.802875,3.087606,37.850000,-122.25000
...,...,...,...,...,...,...,...,...
20635,1.560300,25.000000,5.045455,1.133333,1419.802875,2.560606,35.642578,-119.57379
20636,3.850569,18.000000,5.433694,1.315789,356.000000,3.122807,35.642578,-121.21000
20637,3.850569,17.000000,5.205543,1.120092,1007.000000,2.325635,39.430000,-119.57379
20638,1.867200,28.616222,5.433694,1.171920,741.000000,3.087606,39.430000,-121.32000


In [62]:
imp_0 = SimpleImputer(missing_values=np.nan,strategy="constant",fill_value=0)
# Prepare 0 for missing values
x_missing_0 = imp_0.fit_transform(x_missing)
pd.DataFrame(x_missing_0)

,0,1,2,3,4,5,6,7
0,8.3252,0.0,6.984127,0.000000,0.0,2.555556,37.88,0.00
1,8.3014,0.0,0.000000,0.971880,2401.0,2.109842,0.00,-122.22
2,0.0000,0.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,0.0,5.817352,1.073059,0.0,2.547945,37.85,-122.25
4,3.8462,0.0,0.000000,1.081081,0.0,0.000000,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,0.0,2.560606,0.00,0.00
20636,0.0000,18.0,0.000000,1.315789,356.0,3.122807,0.00,-121.21
20637,0.0000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,0.00
20638,1.8672,0.0,0.000000,1.171920,741.0,0.000000,39.43,-121.32
